# Advanced TTS Pipeline (V2)
## Step 0: Setup & Logging
Initializes the run logger to track all outputs to `Runs/{Date}/log.txt`.

In [2]:
import sys
import os

# Add src to path
if os.path.abspath('src') not in sys.path:
    sys.path.append(os.path.abspath('src'))
if os.path.abspath('.') not in sys.path:
    sys.path.append(os.path.abspath('.'))

from src.logger_utils import run_logger

# Start the logging session for this run
output_dir = run_logger.start_run()
print(f"Output directory set to: {output_dir}")

SyntaxError: (unicode error) 'utf-8' codec can't decode byte 0xff in position 0: invalid start byte (__init__.py, line 1)

## Step 1: Configuration
Set up API keys and parameters.

In [ ]:
import os

# === 🛠️ USER CONFIGURATION ===

# 1. FILES
EPUB_PATH = "data/The Last Experience Point - Chapters 1 to 10.epub"
CONFIG_PATH = "data/book_config_tlpoe.json"

# 2. AI SETTINGS
MODEL_NAME = "google/gemini-2.5-flash"
MAX_RETRIES = 3         # Set to 0 to disable auto-retries
MAX_COST_PER_RUN = 2.0  # Safe stop limit ($)

# 3. BATCH PROCESSING
BATCH_START = 3         # First Chapter to process
BATCH_END = 10          # Last Chapter to process

# 4. API KEYS (Set here or in .env)
# os.environ["OPENROUTER_API_KEY"] = "sk-or-..." 


## Step 2: EPUB Processing
Load and clean the EPUB file.

In [ ]:
import json
from book_processor import EpubLoader, Chapter
from llm_handler import LLMHandler

# EXAMPLE INPUT - Replace with your valid path
epub_path = EPUB_PATH

if os.path.exists(epub_path):
    loader = EpubLoader(epub_path)
    chapters = loader.extract_chapters()
    print(f"Loaded {len(chapters)} chapters.")
    if chapters:
        print(f"Preview Chapter 1: {chapters[0].title}")
else:
    print(f"File not found: {epub_path}. Please place an epub file in the directory.")

## Step 3: Run AI Processing
Initialize the LLM handler and process the first chapter as a test.

In [ ]:
from book_processor import EpubLoader, Chapter
from llm_handler import LLMHandler
from pipeline_utils import CostMonitor
from character_manager import CharacterManager
from dotenv import load_dotenv
import json
from tqdm import tqdm

# Load Environment
load_dotenv()

# CONFIG
config_path = CONFIG_PATH
char_manager = CharacterManager(config_path)

# Initialize LLM
# Priority: Local Var > OpenRouter Env > OpenAI Env
if 'api_key' not in locals():
    api_key = os.getenv("OPENROUTER_API_KEY") or os.getenv("OPENAI_API_KEY") or "YOUR_API_KEY_HERE"

# Ensure Output Dir (Standalone safety)
if 'output_dir' not in locals():
    print('Warning: running standalone. Saving to current dir.')
    output_dir = '.'
else:
    os.makedirs(output_dir, exist_ok=True)

handler = LLMHandler(api_key=api_key, model=MODEL_NAME)

# --- SAFETY CONFIG ---
MAX_COST_PER_RUN = MAX_COST_PER_RUN
try:
    start_usage = CostMonitor.get_usage(api_key)
    print(f"Initial Usage: ${start_usage:.4f}")
except Exception:
    start_usage = 0.0
    print("Initial Usage: Unknown (monitor failed)")

# --- BATCH CONFIGURATION ---
START_IDX = BATCH_START
END_IDX = BATCH_END
# ---------------------------

if 'chapters' in locals() and chapters:
    print(f"Starting Batch Run: Chapters {START_IDX} to {END_IDX}")
    effective_end = min(END_IDX + 1, len(chapters))
    
    for i in tqdm(range(START_IDX, effective_end), desc="Processing Chapters", unit="chapter", ascii=" ░▒▓█", ncols=100):
        # Cost Check
        current_usage = CostMonitor.get_usage(api_key)
        run_cost = current_usage - start_usage
        if run_cost > MAX_COST_PER_RUN:
            print(f"\n[SAFETY] Limit Reached! Run Cost: ${run_cost:.4f}")
            break
            
        chapter = chapters[i]
        
        # Safe Filename
        safe_title = "".join([c if c.isalnum() else "_" for c in chapter.title])[:50]
        filename = f"chapter_{i:03d}_{safe_title}.json"
        filepath = os.path.join(output_dir, filename)
        
        if os.path.exists(filepath):
            print(f"Skipping Chapter {i}: {filename} exists.")
            continue
            
        print(f"\n=== Processing Chapter {i}: {chapter.title} ===")
        try:
            # STEP 1: CONTEXT AWARE PROMPTING (Local)
            context_header = char_manager.get_dynamic_prompt_header(chapter.content_raw)
            
            # STEP 2: GENERATION (API Call 1)
            segments = handler.process_chapter(
                chapter.content_raw,
                context_header=context_header,
                chapter_id=i,
                max_retries=MAX_RETRIES
            )
            
            # STEP 3: AUTOSYNC (Local)
            char_manager.update_profiles(segments, chapter_id=i)
            
            # STEP 4: ACTIVE ENRICHMENT (API Call 2)
            # Deduplicates and profiles new characters
            char_manager.enrich_db(handler.client, segments, handler.model)
            
            # Save
            with open(filepath, "w", encoding="utf-8") as f:
                json.dump([s.model_dump() for s in segments], f, indent=2)
            print(f"Saved: {filename}")
            
        except Exception as e:
            print(f"ERROR processing Chapter {i}: {e}")
            continue
            
    print("\nBatch Run Complete.")